        # Evaluación de escenarios y comparación de alternativas

        **Modelación y Simulación Computacional** · Maestría en Ingeniería ·
        Universidad de Sucre · periodo 2026-2

        **Unidad 3.** Simulación de sistemas y análisis de escenarios ·
        **Subtema del plan 3.3**

        Autor, Prof. Daniel Otero Meza, Ing., Ph.D.

        <!-- ENLACE_COLAB -->
        [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/<usuario>/<repositorio>/blob/main/03_cuadernos/Unidad3/U3_03_escenarios_y_alternativas.ipynb)

        Ninguna decisión de ingeniería se toma con una corrida. Este cuaderno
construye la matriz de escenarios del Ejemplo 3.5, reduce cada corrida a
dos indicadores en conflicto, extrae la frontera de compromiso sin
recurrir a pesos arbitrarios y termina reemplazando los niveles discretos
por distribuciones continuas mediante el muestreo de Monte Carlo del
Ejemplo 3.6.

        ## Objetivos de aprendizaje

        Al terminar este cuaderno el estudiante debe ser capaz de

        1. Distinguir escenario y corrida según la Definición 3.5 y clasificar los factores en de decisión y de contexto.
2. Construir un barrido factorial completo con `itertools.product` y reproducir la Tabla 3.3 del libro.
3. Extraer la frontera de alternativas no dominadas dentro de un mismo contexto y leer en ella el precio de cada mejora.
4. Propagar la variabilidad de seis entradas por muestreo de Monte Carlo y reportar media, percentiles, probabilidad de excedencia y error estándar.
5. Implementar el muestreo por hipercubo latino con NumPy y medir la reducción de varianza que produce con el mismo número de corridas.

## Puesta a punto

La primera celda detecta el entorno e instala solo lo que falte, de modo que
el cuaderno abre igual en Google Colab y en JupyterLab. La segunda fija la
semilla del curso, la paleta del libro y las funciones auxiliares. La semilla
vale 20262 y ningún resultado depende de una ejecución concreta.

In [ ]:
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict) -> None:
    """Instala solo los paquetes que no estén disponibles."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

COLORES = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.figsize": (9.0, 4.4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "legend.frameon": False})

trapecio = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

pd.set_option("display.width", 110)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def carpeta_datos() -> Path:
    """Ubica la carpeta datos sin usar rutas absolutas.

    Busca hacia arriba desde el directorio de trabajo, de modo que funcione
    tanto en el repositorio como en una sesión de Colab donde el cuaderno se
    abre suelto. Si no la encuentra, la crea junto al cuaderno.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        if (candidata / "datos").is_dir():
            return candidata / "datos"
    destino = base / "datos"
    destino.mkdir(exist_ok=True)
    return destino


def leer_datos(nombre: str, respaldo) -> pd.DataFrame:
    """Lee un archivo de datos y lo reconstruye si no está disponible.

    El argumento respaldo es una función sin argumentos que devuelve el
    mismo cuadro de datos, construido con las cifras publicadas en el libro.
    Así el cuaderno nunca depende de una descarga.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        ruta = candidata / "datos" / nombre
        if ruta.exists():
            return pd.read_csv(ruta)
    tabla = respaldo()
    tabla.to_csv(carpeta_datos() / nombre, index=False)
    return tabla


def comparar(etiqueta: str, calculado: float, libro: float,
             tol: float, unidad: str = "") -> bool:
    """Imprime y verifica un valor calculado frente al que publica el libro."""
    dif = abs(calculado - libro)
    ok = dif <= tol
    marca = "coincide" if ok else "NO coincide"
    print(f"{etiqueta:<46s} calculado {calculado:>14.6g} {unidad:<12s}"
          f" libro {libro:>12.6g}   {marca}")
    return ok


print("semilla del curso", SEMILLA)

In [ ]:
def respaldo_valores_libro() -> pd.DataFrame:
    """Cifras publicadas en el capítulo 3, transcritas del libro."""
    filas = [
    ("colebrook_velocidad", 1.6977, "m/s", "Ejemplo 3.1"),
    ("colebrook_reynolds", 507267.0, "adimensional", "Ejemplo 3.1"),
    ("colebrook_rugosidad_relativa", 0.0008667, "adimensional", "Ejemplo 3.1"),
    ("colebrook_factor_friccion", 0.0196228, "adimensional", "Ejemplo 3.1"),
    ("colebrook_perdida_carga", 8.167, "m", "Ejemplo 3.1"),
    ("colebrook_swamee_jain", 0.019742, "adimensional", "Ejemplo 3.1"),
    ("colebrook_orden_newton", 2.0, "adimensional", "Ejemplo 3.1"),
    ("lagunas_perfil_1", 142.42, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_2", 83.4, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_3", 41.5, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_4", 17.65, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_5", 7.33, "mg/L", "seccion 3.1.2"),
    ("lagunas_remocion", 97.07, "por ciento", "seccion 3.1.2"),
    ("lagunas_retencion", 8.33, "d", "seccion 3.1.2"),
    ("lagunas_carga_afluente", 300000.0, "mg/d", "seccion 3.1.2"),
    ("lagunas_carga_efluente", 8792.84, "mg/d", "seccion 3.1.2"),
    ("lagunas_consumo", 291207.16, "mg/d", "seccion 3.1.2"),
    ("fermentador_tiempo_25C", 17.0604614, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_30C", 10.9186953, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_35C", 7.5824273, "h", "Ejemplo 3.2"),
    ("fermentador_invariante", 12.5, "g/L", "Ejemplo 3.2"),
    ("fermentador_mumax_25C", 0.192, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_30C", 0.3, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_35C", 0.432, "1/h", "Ejemplo 3.2"),
    ("fermentador_evaluaciones", 584.0, "evaluaciones", "Ejemplo 3.2"),
    ("tolerancia_tiempo_rtol3", 10.9028, "h", "seccion 3.2.1"),
    ("tolerancia_error_rtol3", 0.00146, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol3", 44.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol6", 1.85e-07, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol6", 194.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol9", 2.45e-10, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol9", 584.0, "evaluaciones", "seccion 3.2.1"),
    ("circuito_autovalor_rapido", -1005.0002, "1/s", "Ejemplo 3.3"),
    ("circuito_autovalor_lento", -0.0497512, "1/s", "Ejemplo 3.3"),
    ("circuito_razon_rigidez", 20200.0, "adimensional", "Ejemplo 3.3"),
    ("circuito_pasos_rk45", 30376.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_rk45", 212576.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_pasos_bdf", 144.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_bdf", 292.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_paso_medio_rk45", 0.003292, "s", "Ejemplo 3.3"),
    ("circuito_tau_rapida", 0.000995, "s", "Ejemplo 3.3"),
    ("circuito_tau_lenta", 20.1, "s", "Ejemplo 3.3"),
    ("circuito_error_rk45", 9.1e-07, "adimensional", "Ejemplo 3.3"),
    ("circuito_error_bdf", 1.1e-06, "adimensional", "Ejemplo 3.3"),
    ("circuito_producto_h_lambda", 3.31, "adimensional", "Ejemplo 3.3"),
    ("rio_peclet_celda", 0.583, "adimensional", "Ejemplo 3.4"),
    ("rio_pico_analitico", 1.8655, "mg/L", "Ejemplo 3.4"),
    ("rio_abscisa_pico", 2460.0, "m", "Ejemplo 3.4"),
    ("rio_error_maximo", 7.18e-06, "kg/m3", "Ejemplo 3.4"),
    ("rio_masa_remanente", 24.740935, "kg", "Ejemplo 3.4"),
    ("rio_orden_observado", 2.0, "adimensional", "Ejemplo 3.4"),
    ("rio_paso_difusion", 16.67, "s", "seccion 3.3.2"),
    ("rio_paso_adveccion", 57.14, "s", "seccion 3.3.2"),
    ("rio_error_explicito_d045", 6.3e-05, "kg/m3", "Ejemplo 3.4"),
    ("riego_frontera_bruto_p045_d45", 393.8, "mm", "Ejemplo 3.5"),
    ("riego_frontera_bruto_p085_d65", 243.8, "mm", "Ejemplo 3.5"),
    ("riego_deficit_p085_d65", 21.5, "por ciento", "Ejemplo 3.5"),
    ("riego_no_dominadas_clima_normal", 5.0, "alternativas", "Ejemplo 3.5"),
    ("biogas_desviacion_replicas_mc", 0.933, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion_replicas_lhs", 0.112, "kW h/d", "Ejemplo 3.6"),
    ("biogas_reduccion_varianza", 69.0, "veces", "Ejemplo 3.6"),
    ("riego_agua_aprovechable", 126.0, "mm", "Ejemplo 3.5"),
    ("biogas_media", 92.34, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion", 21.22, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p05", 61.71, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p50", 90.05, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p95", 130.62, "kW h/d", "Ejemplo 3.6"),
    ("biogas_excedencia_110", 0.1945, "adimensional", "Ejemplo 3.6"),
    ("biogas_error_estandar", 0.15, "kW h/d", "Ejemplo 3.6"),
    ("biogas_valores_centrales", 91.53, "kW h/d", "Ejemplo 3.6"),
    ("lcoe_crf", 0.101806, "1/a", "Ejemplo 3.7"),
    ("lcoe_factor_degradacion", 0.931205, "adimensional", "Ejemplo 3.7"),
    ("lcoe_produccion_especifica", 1325.6, "kW h/(kW a)", "Ejemplo 3.7"),
    ("lcoe_nominal", 0.083523, "USD/(kW h)", "Ejemplo 3.7"),
    ("lcoe_elasticidad_inversion", 0.87355, "adimensional", "Ejemplo 3.7"),
    ("lcoe_elasticidad_tasa", 0.637, "adimensional", "Ejemplo 3.7"),
    ("lcoe_amplitud_tasa", 35.8, "por ciento", "Ejemplo 3.7"),
    ("lcoe_amplitud_irradiacion", 16.1, "por ciento", "Ejemplo 3.7"),
    ("ishigami_s1", 0.3138, "adimensional", "seccion 3.6.2"),
    ("ishigami_s2", 0.4423, "adimensional", "seccion 3.6.2"),
    ("ishigami_s3", -0.0001, "adimensional", "seccion 3.6.2"),
    ("ishigami_st3", 0.2436, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_s_B0", 0.616, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_suma_primer_orden", 0.985, "adimensional", "seccion 3.6.2"),
    ("morris_evaluaciones", 210.0, "evaluaciones", "seccion 3.6.2"),
    ]
    return pd.DataFrame(filas, columns=["clave", "valor", "unidad", "referencia"])


LIBRO = leer_datos("valores_libro_cap3.csv",
                   respaldo_valores_libro).set_index("clave")["valor"]
print(f"cifras del libro disponibles, {LIBRO.size} registros")

## 1. Escenario, corrida y costo combinatorio

Una corrida es una ejecución del modelo con un juego completo de valores
de entrada. Un escenario es de nivel superior, pues describe de manera
coherente una situación posible y se traduce en uno o varios juegos de
valores. La Definición 3.5 del libro separa además los factores de
decisión, que el ingeniero controla, de los de contexto, que sufre y no
elige, y esa distinción determina cómo se interpreta la comparación. Las
alternativas se comparan dentro de un mismo contexto y nunca a través de
contextos distintos.

El costo del barrido completo crece como el producto de los niveles de
cada factor. El problema 3-20 pide ese cálculo para cinco factores de
tres niveles con corridas de 45 s.

In [ ]:
import itertools

for factores in (3, 5, 7, 10):
    corridas = 3**factores
    horas = corridas * 45 / 3600
    print(f"{factores:2d} factores de tres niveles  ->  {corridas:6,d} corridas"
          f"   {horas:8.2f} h a 45 s por corrida")
print()
print("un diseño factorial fraccionado de resolución cuatro sobre siete")
print("factores usa 16 corridas en lugar de 2187 y estima los efectos")
print("principales confundiendo solo interacciones de orden alto")

## 2. El modelo, balance hídrico del suelo

El Ejemplo 3.5 cultiva maíz de 90 días de ciclo en un suelo franco con
humedad a capacidad de campo de 0.29 y punto de marchitez de 0.15, sobre
una profundidad radicular efectiva de 0.90 m. El agua aprovechable total
vale \(\mathrm{TAW} = 1000(\theta_{cc}-\theta_{pm})Z_r = 126\) mm y el
agua fácilmente aprovechable \(\mathrm{RAW} = 0.55\,\mathrm{TAW}\). El
agotamiento se actualiza cada día con la Ecuación 3.19 del libro y el
riego se dispara cuando el agotamiento alcanza la fracción \(p\) del
agua aprovechable.

El capítulo no publica la curva de coeficiente de cultivo, de modo que el
cuaderno adopta una curva trapezoidal de la publicación 56 de la FAO,
guardada en `datos/kc_maiz_90d.csv`, con etapas de 12, 37, 34 y 7 días y
coeficientes de 0.35, 1.15 y 0.40. Esa curva reproduce de manera exacta
las veintisiete láminas brutas de la Tabla 3.3 y los déficits con
diferencia inferior a tres décimas de punto porcentual.

In [ ]:
THETA_CC, THETA_PM, PROFUNDIDAD = 0.29, 0.15, 0.90
FRACCION_SIN_ESTRES, EFICIENCIA = 0.55, 0.80
TAW = 1000 * (THETA_CC - THETA_PM) * PROFUNDIDAD      # mm
RAW = FRACCION_SIN_ESTRES * TAW                        # mm


def respaldo_kc() -> pd.DataFrame:
    """Curva trapezoidal de coeficiente de cultivo para 90 días."""
    etapas, valores = (12, 37, 34, 7), (0.35, 1.15, 0.40)
    largo_ini, largo_des, largo_med, largo_fin = etapas
    kc_ini, kc_med, kc_fin = valores
    curva = [kc_ini] * largo_ini
    curva += [kc_ini + (kc_med - kc_ini) * j / largo_des
              for j in range(1, largo_des + 1)]
    curva += [kc_med] * largo_med
    curva += [kc_med + (kc_fin - kc_med) * j / largo_fin
              for j in range(1, largo_fin + 1)]
    return pd.DataFrame({"dia": np.arange(1, 91),
                         "kc": np.round(curva, 6)})


kc_tabla = leer_datos("kc_maiz_90d.csv", respaldo_kc)
KC = kc_tabla["kc"].to_numpy()

comparar("agua aprovechable total", TAW, LIBRO["riego_agua_aprovechable"],
         1e-9, "mm")
print(f"agua fácilmente aprovechable                   {RAW:.2f} mm")
print(f"suma de la curva de coeficiente de cultivo     {KC.sum():.3f}")
assert KC.size == 90, "la curva debe cubrir los 90 días del ciclo"

### Ejercicio 1

Complete el balance hídrico diario. Para cada día calcule el coeficiente
de estrés con el agotamiento del día anterior, aplique la transpiración
real, dispare el riego cuando el agotamiento alcance la fracción \(p\)
del agua aprovechable, acumule la percolación cuando el agotamiento se
vuelva negativo y limite el agotamiento al agua aprovechable total. La
celda de partida devuelve un resultado fijo, con lo cual las veintisiete
corridas salen iguales.

In [ ]:
# COMPLETE: escriba el bucle diario del balance hídrico.
#   Ks  = min(1, (TAW - Dr)/(TAW - RAW))   con Dr del día anterior
#   ETa = Ks*kc*ETo   y   Dr = Dr + ETa
#   si Dr >= p*TAW, aplique la lámina neta d y cuente un riego
#   si Dr < 0, acumule la percolación y ponga Dr en cero
#   finalmente limite Dr a TAW
REVISAR_BALANCE = False


def balance_hidrico(p: float, d: float, ETo: float,
                    kc: np.ndarray = None) -> dict:
    """Balance hídrico diario del suelo, con láminas en milímetros."""
    kc = KC if kc is None else kc
    return dict(riegos=5, neto=5 * d, bruto=5 * d / EFICIENCIA,
                deficit=0.0, percolacion=0.0, agotamiento=0.0,
                transpiracion=0.0, potencial=float(np.sum(kc) * ETo))

In [ ]:
una = balance_hidrico(0.45, 45.0, 4.8)
for clave, valor in una.items():
    print(f"{clave:<16s} {valor:10.3f}")

cierre = (una["neto"] - una["percolacion"] + una["agotamiento"]
          - una["transpiracion"])
print(f"\ncierre del balance de agua                     {cierre:.6f} mm")
if REVISAR_BALANCE:
    assert abs(cierre) < 0.1, \
        "la lámina neta menos la percolación más el agotamiento final " \
        "debe reproducir la transpiración acumulada"
    print("el balance de agua cierra con error inferior a 0.1 mm, "
          "tal como pide la verificación del Ejemplo 3.5")
else:
    print("complete la celda anterior y ponga REVISAR_BALANCE = True")

## 3. La matriz de escenarios

Se comparan tres umbrales de agotamiento para disparar el riego, 0.45,
0.65 y 0.85, y tres láminas netas por evento, 25, 45 y 65 mm, bajo tres
niveles de demanda climática, 4.0, 4.8 y 5.6 mm/d de evapotranspiración
de referencia. Los dos primeros son factores de decisión y el tercero es
un factor de contexto. El barrido completo produce las veintisiete
corridas de la Tabla 3.3, tal como hace el Listado 3.7 del libro.

In [ ]:
NIVELES = {"p": [0.45, 0.65, 0.85],
           "d": [25.0, 45.0, 65.0],
           "ETo": [4.0, 4.8, 5.6]}

registros = []
for combinacion in itertools.product(*NIVELES.values()):
    escenario = dict(zip(NIVELES, combinacion))
    registros.append({**escenario, **balance_hidrico(**escenario)})
barrido = pd.DataFrame(registros)
barrido["bruto_redondeado"] = np.round(barrido["bruto"], 1)

pivote = barrido.pivot_table(index=["p", "d"], columns="ETo",
                             values=["bruto_redondeado", "deficit"])
print(pivote.to_string(float_format=lambda v: f"{v:8.2f}"))

In [ ]:
def respaldo_tabla_libro() -> pd.DataFrame:
    """Tabla 3.3 del libro, transcrita."""
    filas = [
        (0.45, 25, 4.0, 312.5, 0.00), (0.45, 25, 4.8, 406.2, 0.00),
        (0.45, 25, 5.6, 468.8, 0.00), (0.45, 45, 4.0, 337.5, 0.00),
        (0.45, 45, 4.8, 393.8, 0.00), (0.45, 45, 5.6, 506.2, 0.00),
        (0.45, 65, 4.0, 406.2, 0.00), (0.45, 65, 4.8, 487.5, 0.00),
        (0.45, 65, 5.6, 568.8, 0.00), (0.65, 25, 4.0, 281.2, 4.82),
        (0.65, 25, 4.8, 343.8, 5.09), (0.65, 25, 5.6, 437.5, 4.43),
        (0.65, 45, 4.0, 281.2, 2.27), (0.65, 45, 4.8, 393.8, 3.13),
        (0.65, 45, 5.6, 450.0, 3.04), (0.65, 65, 4.0, 325.0, 2.03),
        (0.65, 65, 4.8, 406.2, 1.63), (0.65, 65, 5.6, 487.5, 2.02),
        (0.85, 25, 4.0, 125.0, 33.48), (0.85, 25, 4.8, 187.5, 36.20),
        (0.85, 25, 5.6, 218.8, 36.68), (0.85, 45, 4.0, 168.8, 25.16),
        (0.85, 45, 4.8, 225.0, 26.62), (0.85, 45, 5.6, 281.2, 27.84),
        (0.85, 65, 4.0, 243.8, 21.71), (0.85, 65, 4.8, 243.8, 21.50),
        (0.85, 65, 5.6, 325.0, 22.28)]
    return pd.DataFrame(filas, columns=["p", "d_mm", "ETo_mm_dia",
                                        "bruto_mm", "deficit_pct"])


libro_34 = leer_datos("tabla_3_3_libro.csv", respaldo_tabla_libro)
cotejo = barrido.merge(libro_34, left_on=["p", "d", "ETo"],
                       right_on=["p", "d_mm", "ETo_mm_dia"])
cotejo["dif_bruto"] = cotejo["bruto_redondeado"] - cotejo["bruto_mm"]
cotejo["dif_deficit"] = cotejo["deficit"] - cotejo["deficit_pct"]

print(f"láminas brutas que coinciden con la Tabla 3.3  "
      f"{int((cotejo['dif_bruto'].abs() < 1e-9).sum())} de 27")
print(f"diferencia máxima en el déficit                "
      f"{cotejo['dif_deficit'].abs().max():.3f} puntos porcentuales")

if REVISAR_BALANCE:
    assert int((cotejo["dif_bruto"].abs() < 1e-9).sum()) == 27, \
        "las veintisiete láminas brutas deben reproducir la Tabla 3.3"
    assert cotejo["dif_deficit"].abs().max() < 0.5, \
        "el déficit debe quedar a menos de medio punto porcentual"
    print("la matriz de escenarios reproduce la Tabla 3.3")
else:
    print("complete el balance hídrico y ponga REVISAR_BALANCE = True")

Conviene declarar la diferencia que queda. Las veintisiete láminas brutas
coinciden de manera exacta con la Tabla 3.3, porque dependen solo del
número de riegos. Los déficits difieren en menos de tres décimas de punto
porcentual, diferencia que proviene de la curva de coeficiente de cultivo
adoptada, que el libro no publica. Cualquier curva con la misma
evapotranspiración acumulada produce el mismo número de riegos y déficits
ligeramente distintos.

## 4. Indicadores en conflicto y frontera de compromiso

Una tabla de corridas no es una recomendación. Los dos indicadores del
Ejemplo 3.5 son la lámina bruta total aplicada, que se quiere pequeña, y
el déficit relativo de transpiración, que también se quiere pequeño, y
entran en conflicto. Ante indicadores en conflicto la tentación es
combinarlos con pesos, procedimiento que traslada la decisión a la
elección de los pesos y la oculta tras una cifra aparentemente objetiva.

La alternativa no requiere pesos. Una alternativa domina a otra cuando no
es peor en ningún indicador y es estrictamente mejor en al menos uno, y
el conjunto de las no dominadas constituye la frontera de compromiso.

### Ejercicio 2

Complete la extracción de la frontera dentro de un contexto fijo. La
celda de partida no descarta nada, con lo cual la frontera resulta ser la
tabla completa.

In [ ]:
# COMPLETE: marque como dominada cada fila para la cual exista otra
# con bruto y deficit no mayores y con al menos uno estrictamente menor.
REVISAR_FRONTERA = False


def frontera_compromiso(tabla: pd.DataFrame,
                        columnas=("bruto", "deficit")) -> pd.DataFrame:
    """Alternativas no dominadas, con ambos indicadores a minimizar."""
    a, b = columnas
    dominado = pd.Series(False, index=tabla.index)
    return tabla[~dominado].sort_values(a)

In [ ]:
normal = barrido.query("ETo == 4.8").reset_index(drop=True)
frente = frontera_compromiso(normal)

print("alternativas no dominadas en el clima normal")
print(frente[["p", "d", "riegos", "bruto", "deficit"]].to_string(
    index=False, formatters={"bruto": "{:8.1f}".format,
                             "deficit": "{:7.2f}".format}))

dominadas = normal.drop(frente.index)
print("\nalternativas dominadas, descartadas sin discusión")
print(dominadas[["p", "d", "bruto", "deficit"]].to_string(
    index=False, formatters={"bruto": "{:8.1f}".format,
                             "deficit": "{:7.2f}".format}))

comparar("alternativas no dominadas en clima normal", len(frente),
         LIBRO["riego_no_dominadas_clima_normal"], 0, "")
if REVISAR_FRONTERA and REVISAR_BALANCE:
    assert len(frente) == 5, "el libro reporta cinco alternativas no dominadas"
    assert not ((frente["p"] == 0.45) & (frente["d"] == 65.0)).any(), \
        "la alternativa de umbral 0.45 y lámina 65 mm resulta dominada"
    print("la regla de dominancia detecta el desperdicio sin intervención "
          "del analista")
else:
    print("complete las celdas anteriores y ponga las banderas en True")

In [ ]:
desperdicio = barrido.query("p == 0.45 and d == 65.0 and ETo == 4.8").iloc[0]
mejor = barrido.query("p == 0.45 and d == 45.0 and ETo == 4.8").iloc[0]
print(f"umbral 0.45 y lámina 65 mm   bruto {desperdicio['bruto']:.1f} mm"
      f"   percolación {desperdicio['percolacion']:.1f} mm"
      f"   déficit {desperdicio['deficit']:.2f} por ciento")
print(f"umbral 0.45 y lámina 45 mm   bruto {mejor['bruto']:.1f} mm"
      f"   percolación {mejor['percolacion']:.1f} mm"
      f"   déficit {mejor['deficit']:.2f} por ciento")
comparar("lámina bruta de la alternativa 0.45 y 45 mm", round(mejor["bruto"], 1),
         LIBRO["riego_frontera_bruto_p045_d45"], 1e-9, "mm")
print("\nla primera aplica más agua y percola el exceso, de modo que "
      "resulta dominada")
print("el libro reporta 35.8 mm de percolación para esa alternativa, "
      f"aquí resultan {desperdicio['percolacion']:.1f} mm con la curva "
      "de coeficiente de cultivo adoptada")

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.4))
for eto, color, marca_pt, etiqueta in ((4.0, COLORES["verde"], "^", "clima húmedo"),
                                       (4.8, COLORES["azul"], "o", "clima normal"),
                                       (5.6, COLORES["naranja"], "s", "clima seco")):
    sub = barrido.query("ETo == @eto")
    ax.plot(sub["bruto"], sub["deficit"], marca_pt, color=color, ms=5.5,
            mfc="none", label=etiqueta)
ax.plot(frente["bruto"], frente["deficit"], "--", color=COLORES["rojo"],
        lw=1.2, label="frontera de compromiso, clima normal")
for _, fila in frente.iterrows():
    ax.annotate(f"p={fila['p']:.2f}, d={fila['d']:.0f}",
                (fila["bruto"], fila["deficit"]), textcoords="offset points",
                xytext=(6, 6), fontsize=8, color=COLORES["gris"])
ax.set_xlabel("lámina bruta aplicada (mm)")
ax.set_ylabel("déficit de transpiración (por ciento)")
ax.set_title("Compromiso entre el agua aplicada y el déficit del cultivo")
ax.legend(fontsize=8.5)
plt.show()

## 5. Cuando las entradas no toman tres valores sino que se distribuyen

El análisis de escenarios responde bien cuando los factores son
decisiones discretas y mal cuando las entradas varían de manera continua
con una frecuencia conocida. El caudal de un río o el potencial
metanogénico de un lote de estiércol no toman tres valores, sino que se
distribuyen. La Definición 3.7 del libro llama método de Monte Carlo a
generar realizaciones independientes del vector de entradas, evaluar el
modelo en cada una y describir la muestra de respuestas.

El Ejemplo 3.6 alimenta un biodigestor de mezcla completa con estiércol
bovino y pide caracterizar la energía eléctrica diaria y la probabilidad
de cubrir una demanda de 110 kW h/d. Las seis entradas y sus
distribuciones están en `datos/entradas_biodigestor.csv`.

In [ ]:
from scipy.stats import lognorm, norm, triang, uniform


def respaldo_entradas() -> pd.DataFrame:
    """Distribuciones de las entradas del biodigestor, del Ejemplo 3.6."""
    filas = [
        ("m", "masa_de_estiercol", "normal", 1200.0, 90.0, 0.0, "kg/d"),
        ("SV", "fraccion_solidos_volatiles", "normal", 0.115, 0.012, 0.0,
         "adimensional"),
        ("B0", "potencial_metanogenico", "lognormal", 0.21, 0.18, 0.0,
         "m3/kg SV"),
        ("k", "constante_de_hidrolisis", "uniforme", 0.10, 0.25, 0.0, "1/d"),
        ("TRH", "tiempo_de_retencion", "triangular", 22.0, 28.0, 35.0, "d"),
        ("eta", "eficiencia_electrica", "triangular", 0.28, 0.32, 0.36,
         "adimensional")]
    return pd.DataFrame(filas, columns=["factor", "descripcion",
                                        "distribucion", "par1", "par2",
                                        "par3", "unidad"])


entradas = leer_datos("entradas_biodigestor.csv", respaldo_entradas)
print(entradas.to_string(index=False))

PCI = 35.8            # MJ/m3, poder calorífico inferior del metano
DEMANDA = 110.0       # kW h/d
FACTORES = list(entradas["factor"])

### Ejercicio 3

El Algoritmo 3.3 del libro separa el muestreo en el hipercubo unitario de
la transformación a las variables físicas, de modo que cambiar de esquema
de muestreo no obliga a tocar el modelo. Complete la transformación con
las funciones cuantílicas de SciPy, tal como hace el Listado 3.9. Recuerde
que la logarítmico normal se parametriza con la desviación del logaritmo,
\(s = \sqrt{\ln(1+\mathrm{CV}^{2})}\), y con la mediana como escala.
La celda de partida usa marginales uniformes en un rango arbitrario.

In [ ]:
# COMPLETE: use norm.ppf, lognorm.ppf, uniform.ppf y triang.ppf.
#   m    normal de media 1200 y desviación 90
#   SV   normal de media 0.115 y desviación 0.012
#   B0   lognormal de mediana 0.21 y coeficiente de variación 0.18
#   k    uniforme entre 0.10 y 0.25, es decir loc 0.10 y scale 0.15
#   TRH  triangular entre 22 y 35 con moda 28
#   eta  triangular entre 0.28 y 0.36 con moda 0.32
REVISAR_TRANSF = False


def transformar(u: np.ndarray) -> dict:
    """Del hipercubo unitario a las variables físicas del biodigestor."""
    return dict(m=1200.0 + 100.0 * (u[:, 0] - 0.5),
                SV=0.115 + 0.02 * (u[:, 1] - 0.5),
                B0=0.21 + 0.05 * (u[:, 2] - 0.5),
                k=0.10 + 0.15 * u[:, 3],
                TRH=22.0 + 13.0 * u[:, 4],
                eta=0.28 + 0.08 * u[:, 5])

In [ ]:
def energia(m, SV, B0, k, TRH, eta):
    """Energía eléctrica diaria en kW h por día."""
    return m * SV * B0 * (1 - np.exp(-k * TRH)) * PCI * eta / 3.6


N_CORRIDAS = 20000
generador = np.random.default_rng(SEMILLA)
E = energia(**transformar(generador.random((N_CORRIDAS, 6))))

resumen = dict(media=E.mean(), desviacion=E.std(ddof=1),
               p05=np.percentile(E, 5), p50=np.percentile(E, 50),
               p95=np.percentile(E, 95),
               excedencia=(E >= DEMANDA).mean(),
               error_estandar=E.std(ddof=1) / np.sqrt(N_CORRIDAS))
for clave, valor in resumen.items():
    print(f"{clave:<16s} {valor:10.4f}")

In [ ]:
ok = [comparar("media de la energía", resumen["media"],
               LIBRO["biogas_media"], 5e-3, "kW h/d"),
      comparar("desviación de la energía", resumen["desviacion"],
               LIBRO["biogas_desviacion"], 5e-3, "kW h/d"),
      comparar("percentil cinco", resumen["p05"], LIBRO["biogas_p05"],
               5e-3, "kW h/d"),
      comparar("mediana", resumen["p50"], LIBRO["biogas_p50"], 5e-3, "kW h/d"),
      comparar("percentil noventa y cinco", resumen["p95"],
               LIBRO["biogas_p95"], 5e-3, "kW h/d"),
      comparar("probabilidad de cubrir 110 kW h/d", resumen["excedencia"],
               LIBRO["biogas_excedencia_110"], 5e-5, ""),
      comparar("error estándar de la media", resumen["error_estandar"],
               LIBRO["biogas_error_estandar"], 5e-4, "kW h/d")]

centrales = dict(m=1200.0, SV=0.115, B0=0.21, k=0.175, TRH=28.0, eta=0.32)
e_central = float(energia(**{k: np.array([v]) for k, v in centrales.items()})[0])
ok.append(comparar("modelo en los valores centrales", e_central,
                   LIBRO["biogas_valores_centrales"], 5e-3, "kW h/d"))
print(f"\nel valor central queda "
      f"{100 * (resumen['media'] - e_central) / resumen['media']:.1f} "
      "por ciento por debajo de la media muestral")

if REVISAR_TRANSF:
    assert all(ok), "la propagación no reproduce el Ejemplo 3.6"
    print("media, percentiles, excedencia y error estándar coinciden "
          "con el Ejemplo 3.6")
else:
    print("complete la celda anterior y ponga REVISAR_TRANSF = True")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0))
tamanos = np.unique(np.round(np.logspace(1, np.log10(N_CORRIDAS), 60)).astype(int))
medias = np.array([E[:n].mean() for n in tamanos])
errores = np.array([E[:n].std(ddof=1) / np.sqrt(n) for n in tamanos])
ax1.fill_between(tamanos, medias - 1.96 * errores, medias + 1.96 * errores,
                 color=COLORES["azul"], alpha=0.18, lw=0)
ax1.plot(tamanos, medias, color=COLORES["azul"], lw=1.2)
ax1.axhline(E.mean(), color=COLORES["rojo"], ls="--", lw=1.0)
ax1.set_xscale("log")
ax1.set_xlabel("número de corridas N")
ax1.set_ylabel("energía media (kW h/d)")
ax1.set_title("Convergencia del estimador")

ax2.hist(E, bins=55, color=COLORES["azul"], alpha=0.55, edgecolor="white",
         lw=0.4)
for q, color, etiqueta in ((5, COLORES["rojo"], "P5"),
                           (50, COLORES["morado"], "P50"),
                           (95, COLORES["rojo"], "P95")):
    v = np.percentile(E, q)
    ax2.axvline(v, color=color, ls="--", lw=1.0)
    ax2.annotate(f"{etiqueta} = {v:.0f}", (v, 0.92), xycoords=("data", "axes fraction"),
                 rotation=90, fontsize=8, color=color, ha="right", va="top")
ax2.axvline(DEMANDA, color=COLORES["verde"], lw=1.2)
ax2.set_xlabel("energía eléctrica diaria (kW h/d)")
ax2.set_ylabel("frecuencia")
ax2.set_title("Distribución de la respuesta")
plt.tight_layout()
plt.show()

## 6. Muestreo por hipercubo latino

El muestreo aleatorio simple desperdicia esfuerzo, porque varias
realizaciones pueden caer muy cerca unas de otras y dejar regiones
enteras sin visitar. La Definición 3.8 del libro divide el recorrido de
cada entrada en \(N\) estratos de igual probabilidad, extrae exactamente
una realización de cada estrato y combina al azar las extracciones de las
distintas entradas.

### Ejercicio 4

Implemente el muestreo por hipercubo latino con NumPy. Para cada columna,
una permutación de los enteros de cero a \(N-1\) asigna un estrato a
cada realización, y un valor uniforme dentro del estrato completa el
punto. La celda de partida devuelve una muestra aleatoria simple, con lo
cual no hay reducción de varianza alguna.

In [ ]:
# COMPLETE: para cada columna j, u[:, j] = (permutacion(n) + uniforme(n))/n
REVISAR_LHS = False


def muestra_lhs(generador, n: int, d: int) -> np.ndarray:
    """Muestra por hipercubo latino sobre el hipercubo unitario."""
    return generador.random((n, d))       # marcador, es muestreo simple

In [ ]:
REPLICAS, POR_REPLICA = 200, 500
semillas = np.random.SeedSequence(SEMILLA).spawn(REPLICAS)

medias_mc = np.array([energia(**transformar(
    np.random.default_rng(s).random((POR_REPLICA, 6)))).mean()
    for s in semillas])
medias_lhs = np.array([energia(**transformar(
    muestra_lhs(np.random.default_rng(s), POR_REPLICA, 6))).mean()
    for s in semillas])

desv_mc = float(medias_mc.std(ddof=1))
desv_lhs = float(medias_lhs.std(ddof=1))
reduccion = (desv_mc / desv_lhs)**2
print(f"desviación del estimador, muestreo simple      {desv_mc:.3f} kW h/d")
print(f"desviación del estimador, hipercubo latino     {desv_lhs:.3f} kW h/d")
print(f"reducción de varianza                          {reduccion:.1f} veces")
print(f"valor teórico sigma sobre raíz de N            "
      f"{resumen['desviacion'] / np.sqrt(POR_REPLICA):.3f} kW h/d")
print()
print(f"el libro reporta {LIBRO['biogas_desviacion_replicas_mc']:.3f} y "
      f"{LIBRO['biogas_desviacion_replicas_lhs']:.3f} kW h/d, con una "
      f"reducción de {LIBRO['biogas_reduccion_varianza']:.0f} veces")
print("la desviación de las réplicas es a su vez una cantidad estimada con "
      "200 valores, con cerca del cinco por ciento de error relativo, y "
      "depende de cómo se deriven los 200 flujos independientes de la semilla")

if REVISAR_LHS and REVISAR_TRANSF:
    assert reduccion > 30, \
        "el hipercubo latino debe reducir la varianza de manera apreciable"
    assert abs(desv_mc - resumen["desviacion"] / np.sqrt(POR_REPLICA)) < 0.15, \
        "la desviación del muestreo simple debe seguir la ley de la raíz"
    print("\nla reducción de varianza confirma la Definición 3.8")
else:
    print("complete las celdas anteriores y ponga las banderas en True")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0))
muestra_simple = np.random.default_rng(SEMILLA).random((60, 6))
muestra_estrat = muestra_lhs(np.random.default_rng(SEMILLA), 60, 6)
ax1.plot(muestra_simple[:, 2], muestra_simple[:, 4], "o", ms=4,
         color=COLORES["gris"], label="muestreo simple")
ax1.plot(muestra_estrat[:, 2], muestra_estrat[:, 4], "s", ms=4,
         color=COLORES["azul"], mfc="none", label="hipercubo latino")
ax1.set_xlabel("cuantil del potencial metanogénico")
ax1.set_ylabel("cuantil del tiempo de retención")
ax1.set_title("Cobertura del hipercubo con 60 puntos")
ax1.legend(fontsize=8.5)

ax2.hist(medias_mc, bins=24, color=COLORES["gris"], alpha=0.55,
         label=f"simple, s = {desv_mc:.3f}")
ax2.hist(medias_lhs, bins=24, color=COLORES["azul"], alpha=0.75,
         label=f"latino, s = {desv_lhs:.3f}")
ax2.axvline(E.mean(), color=COLORES["rojo"], ls="--", lw=1.0)
ax2.set_xlabel("media estimada con 500 corridas (kW h/d)")
ax2.set_ylabel("réplicas")
ax2.set_title("Dispersión del estimador en 200 réplicas")
ax2.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

## 7. Cuántas corridas hacen falta

El Teorema 3.3 del libro afirma que el error del estimador decrece como
\(\sigma/\sqrt{N}\) con independencia de la dimensión de la entrada.
Las dos lecturas apuntan en direcciones opuestas. La tasa no se degrada
al aumentar el número de entradas inciertas, cosa que sí ocurre con las
cuadraturas de malla, pero la tasa es lenta, pues reducir el error a la
décima parte exige multiplicar por cien el número de corridas.

### Ejercicio 5

El problema 3-22 entrega una simulación de 2000 corridas con media de
318 kg/h y desviación de 47 kg/h, y pide el error estándar de la media y
las corridas necesarias para llevar la semiamplitud al 95 por ciento
hasta 1 kg/h. Complete las dos funciones.

In [ ]:
# COMPLETE: error estándar = s/sqrt(N)
#           corridas necesarias = (1.96*s/semiamplitud)**2
REVISAR_TAMANO = False


def error_estandar(desviacion: float, n: int) -> float:
    """Error estándar de la media muestral."""
    return 0.0


def corridas_necesarias(desviacion: float, semiamplitud: float,
                        z: float = 1.96) -> int:
    """Corridas para alcanzar una semiamplitud dada al 95 por ciento."""
    return 1

In [ ]:
ee_problema = error_estandar(47.0, 2000)
n_problema = corridas_necesarias(47.0, 1.0)
print(f"problema 3-22   error estándar con 2000 corridas   {ee_problema:.4f} kg/h")
print(f"                semiamplitud al 95 por ciento      "
      f"{1.96 * ee_problema:.4f} kg/h")
print(f"                corridas para semiamplitud 1 kg/h  {n_problema:,d}")

print(f"\nbiodigestor     error estándar con {N_CORRIDAS} corridas   "
      f"{error_estandar(resumen['desviacion'], N_CORRIDAS):.4f} kW h/d")
print(f"                corridas para semiamplitud 0.1     "
      f"{corridas_necesarias(resumen['desviacion'], 0.1):,d}")
print("\nescalar el cuaderno a esa cifra es cuestión de cambiar N_CORRIDAS, "
      "porque el modelo está vectorizado y el costo crece de forma lineal")

if REVISAR_TAMANO:
    assert abs(ee_problema - 47.0 / np.sqrt(2000)) < 1e-12
    assert n_problema == 8487, \
        "con 1.96 por 47 sobre 1 elevado al cuadrado resultan 8487 corridas"
    print("\nlas dos cifras del problema 3-22 quedan verificadas")
else:
    print("\ncomplete la celda anterior y ponga REVISAR_TAMANO = True")

## 8. Problemas del capítulo

El problema 3-21 pide construir la frontera del clima seco e indicar qué
alternativas cambian de estatus frente al clima normal. La celda siguiente
lo resuelve con la misma función de dominancia.

In [ ]:
seco = barrido.query("ETo == 5.6").reset_index(drop=True)
frente_seco = frontera_compromiso(seco)
etiquetar = lambda t: {(float(f["p"]), float(f["d"])) for _, f in t.iterrows()}
en_normal, en_seco = etiquetar(frente), etiquetar(frente_seco)
nombrar = lambda conjunto: ", ".join(f"p={a:.2f} con d={b:.0f} mm"
                                     for a, b in sorted(conjunto)) or "ninguna"

print("frontera del clima seco")
print(frente_seco[["p", "d", "riegos", "bruto", "deficit"]].to_string(
    index=False, formatters={"bruto": "{:8.1f}".format,
                             "deficit": "{:7.2f}".format}))
print("\nalternativas que entran a la frontera en el clima seco  ",
      nombrar(en_seco - en_normal))
print("alternativas que salen de la frontera                   ",
      nombrar(en_normal - en_seco))

robusta = barrido.query("p == 0.65 and d == 65.0")
print("\numbral 0.65 con lámina de 65 mm en los tres climas")
print(robusta[["ETo", "riegos", "bruto", "deficit"]].to_string(
    index=False, formatters={"bruto": "{:8.1f}".format,
                             "deficit": "{:7.2f}".format}))
print(f"\ndéficit máximo en los tres climas   "
      f"{robusta['deficit'].max():.2f} por ciento")
print(f"riegos máximos en los tres climas   {int(robusta['riegos'].max())}")
print("el libro recomienda esa alternativa con déficit inferior al 2.1 por "
      "ciento, lo cual se confirma, y con cinco riegos o menos, que solo "
      "se cumple en los climas húmedo y normal, pues el clima seco exige seis")

## 9. Cierre

Al terminar este cuaderno el estudiante debe poder hacer lo siguiente.

1. Separar factores de decisión y de contexto y comparar alternativas
   dentro de un mismo contexto. Revise la Definición 3.5 si duda.
2. Construir un barrido factorial completo, estimar su costo y justificar
   cuándo conviene un diseño fraccionado.
3. Extraer la frontera de compromiso sin recurrir a pesos y leer en ella
   el precio de cada mejora. Revise la sección 3.4.1 si el conjunto de no
   dominadas queda vacío o completo.
4. Propagar la variabilidad de varias entradas por Monte Carlo y reportar
   el conjunto mínimo defendible, que incluye el estimador, su error
   estándar, los percentiles, el número de corridas, el esquema de
   muestreo y la semilla. Revise el Algoritmo 3.3 y el Teorema 3.3.
5. Implementar el hipercubo latino y comprobar la reducción de varianza
   con réplicas. Revise la Definición 3.8.

El cuaderno siguiente pregunta de dónde viene la dispersión que aquí se
midió, que es el asunto del análisis de sensibilidad.